In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
name = "Ed Donner"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
print(system_prompt)

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [ ]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [ ]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [ ]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [ ]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1])

In [ ]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Improved Workflow: Iterative Reflection Loop

Previous version's `chat()` function calls `rerun()` **at most once** and then returns the result unconditionally — the rerun answer is **never re-evaluated**. If the rerun also produces a bad answer, it escapes quality control with no further action.

```
generate → evaluate → [fail] → rerun once → return (no second check)
```

### Improvement: `chat_v2`

The upgraded version closes this gap with a **configurable retry loop**:

```
generate → evaluate → [fail] → rerun → evaluate → [fail] → rerun → ... up to MAX_RETRIES
```

Key upgrades:
| Feature | Previous Version `chat()` | Improved `chat_v2()` |
|---|---|---|
| Max retries | 1 (no loop) | Configurable `MAX_RETRIES` (default 3) |
| Re-evaluation after retry | ✗ Never | ✓ Every attempt is evaluated |
| Accumulated feedback | ✗ Only latest | ✓ All past feedback passed to next attempt |
| Exhaustion handling | Returns bad answer silently | Logs warning, returns best available |

In [ ]:
# ============================================================
# IMPROVED WORKFLOW: Iterative Reflection with Retry Loop
# ============================================================
# This version (chat_v2) fixes that with three main changes:
#   1. A retry loop up to MAX_RETRIES — every new answer is
#      re-evaluated before it can be returned.
#   2. Accumulated feedback — all previous rejection reasons are
#      collected and forwarded to the next attempt, so the model
#      gets richer context with each iteration rather than only
#      seeing the most recent failure.
#   3. Exhaustion handling — if all attempts are used up, the
#      last answer is returned with a clear warning log so you
#      know quality control was never satisfied.
# ============================================================

MAX_RETRIES = 3   # How many total attempts the model gets before we give up

def chat_v2(message, history):
    # When the user mentions "patent" we force the model to reply in pig latin,
    # which the evaluator will correctly reject, triggering the retry loop.
    if "patent" in message:
        system = system_prompt + (
            "\n\nEverything in your reply needs to be in pig latin - "
            "it is mandatory that you respond only and entirely in pig latin"
        )
    else:
        system = system_prompt

    # --- Initial generation (attempt 1) ---
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply = response.choices[0].message.content

    # --- Iterative evaluation + retry loop ---
    # feedback_history accumulates every rejection reason across all attempts.
    # This gives the model progressively more context on what to fix.
    feedback_history = []

    for attempt in range(1, MAX_RETRIES + 1):

        # Evaluate the current reply using the Gemini judge from Cell 19
        evaluation = evaluate(reply, message, history)

        if evaluation.is_acceptable:
            # Quality control passed — safe to return
            print(f"Passed evaluation on attempt {attempt} — returning reply")
            return reply

        # --- Attempt failed ---
        feedback_history.append(f"Attempt {attempt}: {evaluation.feedback}")
        print(f"Attempt {attempt} failed evaluation.")
        print(f"  Feedback: {evaluation.feedback}")

        if attempt == MAX_RETRIES:
            # All retries exhausted — break out and return the last answer with a warning
            break

        # --- Build a cumulative system prompt for the next attempt ---
        # Instead of only showing the latest feedback,
        # we inject the FULL history of rejections so the model can avoid
        # repeating the same mistakes it made in earlier attempts.
        updated_system_prompt = system_prompt + "\n\n## Previous answers rejected by quality control\n"
        updated_system_prompt += (
            "You have tried to answer this question multiple times but each attempt "
            "was rejected. Below is the complete history of your attempts and the "
            "feedback you received for each one.\n\n"
        )
        for fb in feedback_history:
            updated_system_prompt += f"- {fb}\n"
        updated_system_prompt += f"\n## Your most recent rejected answer:\n{reply}\n\n"
        updated_system_prompt += (
            "Please write a significantly improved response that specifically addresses "
            "ALL of the feedback points listed above."
        )

        # Generate the next attempt using the enriched system prompt
        retry_messages = (
            [{"role": "system", "content": updated_system_prompt}]
            + history
            + [{"role": "user", "content": message}]
        )
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=retry_messages)
        reply = response.choices[0].message.content

    # --- All attempts exhausted without passing evaluation ---
    # Return the last answer anyway (best effort) but log clearly so you know.
    print(f"Warning: all {MAX_RETRIES} attempts failed evaluation. Returning best available answer.")
    return reply


# Launch the improved chatbot — swap chat_v2 back to chat to compare behaviour
gr.ChatInterface(chat_v2, type="messages").launch()